<a href="https://colab.research.google.com/github/lule-exxeta/pose-estimation/blob/main/notebooks/01_synthetic_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

- Install Isaac Sim


In [ ]:
# Install IsaacSim
%pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cu130
%pip install isaacsim[all,extscache]==5.1.0 --extra-index-url https://pypi.nvidia.com

In [27]:
import os
import numpy as np

In [29]:
import isaacsim
import omni

In [44]:
# Test: Start Isaac Sim app
from isaacsim import SimulationApp
simulation_app = SimulationApp({"headless": True})
simulation_app.close()

TypeError: 'NoneType' object is not callable

## Tasks

- Find USD files
- Create world environment
- Add some USD files with random pose
- Render image and write pose to file  

In [9]:
# List .usd files of EuroBoxes
usd_files = [
    os.path.join("euro-boxes", f)
    for f in os.listdir("euro-boxes")
    if f.endswith(".usd")
]
print("Found", len(usd_files), ".usd files.")

Found 18 .usd files.


In [ ]:
from isaacsim import SimulationApp
from omni.isaac.core import World
from omni.isaac.core import World
from omni.isaac.core.prims import XFormPrim
from omni.isaac.core.utils.stage import add_reference_to_stage
from omni.isaac.core.utils.rotations import euler_angles_to_quat

# Start Isaac Sim
simulation_app = SimulationApp({"headless": True})  # set True for headless

# Create world
world = World(stage_units_in_meters=1.0)
world.scene.add_default_ground_plane()

# Pose
POSITION_RANGE = {
    "x": (-0.5, 0.5),
    "y": (-0.5, 0.5),
    "z": (0.0, 0.5),
}

ROTATION_RANGE = {
    "roll": (0, 360),
    "pitch": (0, 360),
    "yaw": (0, 360),
}

def random_pose():
    pos = np.array([
        np.random.uniform(*POSITION_RANGE["x"]),
        np.random.uniform(*POSITION_RANGE["y"]),
        np.random.uniform(*POSITION_RANGE["z"]),
    ])

    rot = np.deg2rad([
        np.random.uniform(*ROTATION_RANGE["roll"]),
        np.random.uniform(*ROTATION_RANGE["pitch"]),
        np.random.uniform(*ROTATION_RANGE["yaw"]),
    ])

    quat = euler_angles_to_quat(rot)

    return pos, quat


# Add objects
objects = []

for i, usd_path in enumerate(usd_files):
    prim_path = f"/World/Object_{i}"

    add_reference_to_stage(usd_path=usd_path, prim_path=prim_path)

    obj = XFormPrim(prim_path=prim_path, name=f"object_{i}")
    world.scene.add(obj)

    objects.append(obj)


# Simulation loop

world.reset()
pose_records = []

NUM_POSES_PER_OBJECT = 5
for obj_idx, obj in enumerate(objects):
    for pose_idx in range(NUM_POSES_PER_OBJECT):

        position, orientation = random_pose()

        obj.set_world_pose(position=position, orientation=orientation)

        # Step simulation to update physics/render
        for _ in range(10):
            world.step(render=True)

        # Get pose
        pos, quat = obj.get_world_pose()

        record = {
            "object_id": obj_idx,
            "pose_id": pose_idx,
            "position": pos.tolist(),
            "quaternion": quat.tolist(),
        }
        pose_records.append(record)


# TODO: output

simulation_app.close()